In [24]:
#LOAD & CHUNK PDFs

import os
import glob

DATA_DIR = os.path.expanduser("~/Desktop/projects-learning/PillWise/data/raw_pdfs")

pdf_paths = glob.glob(os.path.join(DATA_DIR, "**/*.pdf"), recursive = True)
pdf_paths.sort()

print(f"total PDFs found: {len(pdf_paths)}")

total PDFs found: 30


In [25]:
import fitz

doc = fitz.open(pdf_paths[0])
print(f"file : {pdf_paths[0]}")
print(f"pages : {len(doc)}")
print(f"sample:")
print(doc[0].get_text()[:500])

file : /Users/maneeshari/Desktop/projects-learning/PillWise/data/raw_pdfs/acetaminophen/acetaminophen_1.pdf
pages : 5
sample:
NIGHTTIME COLD AND FLU- acetaminophen, dextromethorphan hbr,
doxylamine succinate solution  
Amazon.com Services LLC
----------
Amazon Basics Nighttime Cold & Flu Drug Facts
Active ingredients (in each 30 mL)
Acetaminophen 650 mg
Dextromethorphan HBr 30 mg
Doxylamine succinate 12.5 mg
Purpose
Pain reliever/fever reducer
Cough suppressant
Antihistamine
Uses
temporarily relieves common cold/flu symptoms:
•
•
•
•
•
•
Warnings
Liver warning: This product contains acetaminophen. Severe liver damage m


In [26]:
for i, page in enumerate(doc):
    text = page.get_text()
    print(f"-- page {i+1} --")
    print(text[:300])
    print()

-- page 1 --
NIGHTTIME COLD AND FLU- acetaminophen, dextromethorphan hbr,
doxylamine succinate solution  
Amazon.com Services LLC
----------
Amazon Basics Nighttime Cold & Flu Drug Facts
Active ingredients (in each 30 mL)
Acetaminophen 650 mg
Dextromethorphan HBr 30 mg
Doxylamine succinate 12.5 mg
Purpose
Pain r

-- page 2 --
Sore throat warning: If sore throat is severe, persists for more than 2 days, is
accompanied or followed by fever, headache, rash, nausea, or vomiting, consult a
doctor promptly.
Do not use
•
•
•
Ask a doctor before use if you have
•
•
•
•
•
•
•
Ask a doctor or pharmacist before use if you are
•
•
W

-- page 3 --
Keep out of reach of children.
Overdose warning: In case of overdose, get medical help or contact a Poison Control
Center right away (1-800-222-1222). Quick medical attention is critical for adults as well
as for children even if you do not notice any signs or symptoms.
Directions
•
•
•
adults & chi

-- page 4 --
12 FL OZ (355 mL)
NIGHTTIME COLD AND FLU  

In [27]:
def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text.strip()

#test
sample_text = extract_text(pdf_paths[0])
print(f"total chars: {len(sample_text)}")
print(sample_text[:300])

total chars: 5602
NIGHTTIME COLD AND FLU- acetaminophen, dextromethorphan hbr,
doxylamine succinate solution  
Amazon.com Services LLC
----------
Amazon Basics Nighttime Cold & Flu Drug Facts
Active ingredients (in each 30 mL)
Acetaminophen 650 mg
Dextromethorphan HBr 30 mg
Doxylamine succinate 12.5 mg
Purpose
Pain r


In [28]:
documents = []

for path in pdf_paths:
    text = extract_text(path)
    drug_name = path.split("/")[-2]
    documents.append({
        "text": text,
        "source": path,
        "drug": drug_name
    })

print(f"total documents loaded: {len(documents)}")
print(f"sample keys: {documents[0].keys()}")
print(f"first drug: {documents[0]['drug']}")


total documents loaded: 30
sample keys: dict_keys(['text', 'source', 'drug'])
first drug: acetaminophen


In [29]:
def fixed_size_chunks(text, chunk_size = 500, overlap = 50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start : end]
        chunks.append(chunk)
        start =  end - overlap
    return chunks

chunks_fixed = fixed_size_chunks(documents[0]['text'])
print(f"total chunks : {len(chunks_fixed)}")
print(f"chunk 1 size : {len(chunks_fixed[0])}")
print(f"chunk 2 size : {len(chunks_fixed[1])}")
print(f"\nend of chunk 1  : ...{chunks_fixed[0][-50:]}")
print(f"start of chunk 2: {chunks_fixed[1][:50]}...")

total chunks : 13
chunk 1 size : 500
chunk 2 size : 500

end of chunk 1  : ...duct contains acetaminophen. Severe liver damage m
start of chunk 2: duct contains acetaminophen. Severe liver damage m...


In [30]:
def sentence_chunks(text, max_sentences = 5):
    import re
    sentences = re.split(r'(?<=[.?!])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i + max_sentences])
        chunks.append(chunk)
    return chunks

chunks_sentences = sentence_chunks(documents[0]['text'])
print(f"total chunks: {len(chunks_sentences)}")
print(f"\nchunk 1:\n{chunks_sentences[0]}")
print(f"\nchunk 2:\n{chunks_sentences[1]}")

total chunks: 4

chunk 1:
NIGHTTIME COLD AND FLU- acetaminophen, dextromethorphan hbr,
doxylamine succinate solution  
Amazon.com Services LLC
----------
Amazon Basics Nighttime Cold & Flu Drug Facts
Active ingredients (in each 30 mL)
Acetaminophen 650 mg
Dextromethorphan HBr 30 mg
Doxylamine succinate 12.5 mg
Purpose
Pain reliever/fever reducer
Cough suppressant
Antihistamine
Uses
temporarily relieves common cold/flu symptoms:
•
•
•
•
•
•
Warnings
Liver warning: This product contains acetaminophen. Severe liver damage may occur
if you take
•
•
•
Allergy alert: Acetaminophen may cause severe skin reactions. Symptoms may
include:
•
•
•
cough due to minor throat and bronchial irritation
sore throat
headache
minor aches and pains
fever
runny nose and sneezing
more than 4,000 mg of acetaminophen in 24 hours
with other drugs containing acetaminophen
3 or more alcoholic drinks every day while using this product
skin reddening
blisters
rash
If a skin reaction occurs, stop use and seek medical

In [31]:

print("── strategy 1: fixed size ──")
print(f"chunks        : {len(chunks_fixed)}")
print(f"avg chunk size: {sum(len(c) for c in chunks_fixed) // len(chunks_fixed)} chars")

print("\n── strategy 2: sentence based ──")
print(f"chunks        : {len(chunks_sentences)}")
print(f"avg chunk size: {sum(len(c) for c in chunks_sentences) // len(chunks_sentences)} chars")

── strategy 1: fixed size ──
chunks        : 13
avg chunk size: 477 chars

── strategy 2: sentence based ──
chunks        : 4
avg chunk size: 1399 chars


## Experiment 1: Chunking Strategy Comparison (single document)

| metric         | fixed size | sentence based |
|----------------|------------|----------------|
| total chunks   | 13         | 5              |
| avg chunk size | 460 chars  | 1082 chars     |

observation: sentence chunking produces larger chunks on medical labels
because bullet points lack sentence-ending punctuation. fixed size chunking
gives more uniform, retrieval-friendly chunks for this document format.

In [32]:
all_chunks_fixed = []
all_chunks_sentence = []

for doc in documents:
    text = doc['text']
    drug = doc['drug']
    source = doc['source']
    
    for chunk in fixed_size_chunks(text):
        all_chunks_fixed.append({
            "chunk": chunk,
            "drug": drug,
            "source": source,
            "strategy": "fixed"
        })
    
    for chunk in sentence_chunks(text):
        all_chunks_sentence.append({
            "chunk": chunk,
            "drug": drug,
            "source": source,
            "strategy": "sentence"
        })

print(f"fixed size chunks   : {len(all_chunks_fixed)}")
print(f"sentence chunks     : {len(all_chunks_sentence)}")

fixed size chunks   : 3139
sentence chunks     : 1609


In [33]:
import pandas as pd

df_fixed = pd.DataFrame(all_chunks_fixed)
df_sentence = pd.DataFrame(all_chunks_sentence)

df_fixed['chunk_len'] = df_fixed['chunk'].apply(len)
df_sentence['chunk_len'] = df_sentence['chunk'].apply(len)

print("── fixed size ──")
print(df_fixed['chunk_len'].describe())

print("\n── sentence based ──")
print(df_sentence['chunk_len'].describe())

── fixed size ──
count    3139.000000
mean      497.606563
std        27.846073
min         6.000000
25%       500.000000
50%       500.000000
75%       500.000000
max       500.000000
Name: chunk_len, dtype: float64

── sentence based ──
count    1609.000000
mean      872.490988
std       499.677559
min        29.000000
25%       597.000000
50%       762.000000
75%      1003.000000
max      6904.000000
Name: chunk_len, dtype: float64


## Experiment 1 results: chunk size distribution across 30 documents

| metric | fixed size | sentence based |
|--------|------------|----------------|
| count  | 3213       | 1626           |
| mean   | 497 chars  | 883 chars      |
| std    | 29         | 499            |
| min    | 6          | 46             |
| max    | 500        | 6904           |

conclusion: fixed size chunking is far more consistent for medical PDF format.
sentence chunking struggles due to bullet points lacking sentence-ending punctuation,
producing chunks up to 6904 chars, too large for effective retrieval.